# PCA reconstruction ablation

This mirrors TEP notebook 4.1. PCA is fit on the three endogenous process variables from healthy training data, and their squared reconstruction error (SPE) is used as the anomaly score. The moving-average window and consecutive-alarm count are fixed to six and five, respectively; the threshold is selected from the healthy phase of faulty validation runs. Test runs do not affect fitting or threshold selection.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from IPython.display import display
from carl.causalai_baselines import run_ablation_comparison

ablation_results = run_ablation_comparison("pca", PROJECT_ROOT)
display(ablation_results["selected_settings"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold
0,pca,spe,2,6,5,2.360392


# 2. Select Ablation Threshold

PCA reconstruction error is smoothed with a six-sample moving average. Its threshold is the maximum pre-fault score across faulty validation runs.

In [2]:
display(ablation_results["validation_metrics"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Anomaly Detection Delay
0,pca,spe,2,6,5,2.360392,0.353811,0.199533,0.0,0.332685,0.844444,0.688889,0.0,0.815789,155,225,49.0


# 3. Test on Faulty Time Series (Ablation Settings)

Evaluate the PCA ablation settings on every faulty held-out test run.

In [3]:
display(ablation_results["faulty_test_metrics"])
display(ablation_results["faulty_test_by_type"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Anomaly Detection Delay
0,pca,spe,2,6,5,2.360392,0.371297,0.221194,0.0,0.362259,0.848889,0.697778,0.0,0.82199,157,225,52.0


,Fault Type,Runs,Detected_Runs,Run_Detection_Rate,Median_Detection_Delay
0,hydraulic_coupling_shift,75,14,0.186667,301.5
1,reaction_coupling_shift,75,74,0.986667,49.5
2,thermal_coupling_shift,75,69,0.920000,45.0


# 4. Test on Fault-Free Time Series (Ablation Settings)

Evaluate the same PCA settings on all healthy held-out test runs.

In [4]:
display(ablation_results["healthy_test_metrics"])
display(ablation_results["comparison"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase FPR,False Alarm Runs,Total Healthy Runs
0,pca,spe,2,6,5,2.360392,1.0,0.0,0.0,0.0,0.0,0,75


,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Faulty Test Sample F1,Faulty Test Trajectory-Phase F1,Faulty Test Trajectory-Phase TPR,Faulty Test Median Detection Delay,Healthy Test Sample FPR,Healthy Test Trajectory-Phase FPR
0,full_var,mahalanobis,2,6,5,3.429900,0.566890,0.891626,0.804444,31.0,0.0,0.0
1,pca,spe,2,6,5,2.360392,0.362259,0.821990,0.697778,52.0,0.0,0.0
